In [4]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np
import os
import torchao
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig
import time


In [5]:
# Load data
bt_sentences = pd.read_csv("C:/Users/rober/UCD/pol42050/new-sound-of-politics/scripts_not_render/bundestag_sentences20_dpi_api.csv")
# bt_sentences = pd.read_csv("bt_speeches_subset.csv")

In [6]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("luerhard/PopBERT", token = os.getenv("HF_TOKEN"))
model = AutoModelForSequenceClassification.from_pretrained("luerhard/PopBERT")
# CPU inference only
device = torch.device("cpu")
model = model.to(device)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [7]:


# torch.set_num_threads(6)
# # Dynamic INT8 quantization for linear layers
# quantized_model = torch.quantization.quantize_dynamic(
#     model,
#     {torch.nn.Linear},
#     dtype=torch.qint8
# )
# quantized_model.eval()


torch.set_num_threads(6)

quantized_model = model
quantize_(quantized_model, Int8DynamicActivationInt8WeightConfig())
quantized_model.eval()


c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\uintx\plain_layout.py:82: UserWarning: Deprecation: PlainAQTTensorImpl is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\affine_quantized_tensor.py:116: UserWarning: Deprecation: AffineQuantizedTensor is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31102, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True, weight=LinearActivationQuantizedTensor(activation=<function _int8_symm_per_token_reduced_range_quant at 0x000001F66CDCADE0>, weight=AffineQuantizedTensor(shape=torch.Size([1024, 1024]), block_size=(1, 1024), device=cpu, _layout=PlainLayout(), tensor_impl_dtype=torch.int8, quant_min=None, quant_max=None)))
              (key): Linear(in_features=1024, out_features=1024, bias=T

In [11]:

torch.set_num_threads(os.cpu_count())

BATCH_SIZE = 128
# Sort by length to minimise padding

subset = bt_sentences#.sample(1000, random_state=69).copy()
subset = subset.sort_values(by="sentence", key=lambda x: x.str.len()).reset_index(drop=True)

sentences = subset["sentence"].tolist()
metadata = list(zip(subset["speech_id"].tolist(), subset["sentence_position"].tolist()))

sentence_scores = []
start = time.time()

for batch_start in range(0, len(sentences), BATCH_SIZE):
    batch_sentences = sentences[batch_start:batch_start + BATCH_SIZE]
    batch_metadata = metadata[batch_start:batch_start + BATCH_SIZE]

    encodings = tokenizer(
        batch_sentences,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    encodings = {k: v.to(device) for k, v in encodings.items()}

    with torch.inference_mode():
        out = quantized_model(**encodings)

    probs = torch.sigmoid(out.logits).cpu().numpy()

    for j, (speech_id, sentence_position) in enumerate(batch_metadata):
        sentence_scores.append([speech_id, sentence_position] + probs[j].tolist())

print(f"Done in {time.time() - start:.1f}s")

c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\utils.py:89: UserWarning: Deprecation: PlainLayout is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\uintx\plain_layout.py:82: UserWarning: Deprecation: PlainAQTTensorImpl is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(
c:\Users\rober\UCD\pol42050\new-sound-of-politics\.venv\Lib\site-packages\torchao\dtypes\affine_quantized_tensor.py:116: UserWarning: Deprecation: AffineQuantizedTensor is deprecated and will be removed in a future release of torchao, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


Done in 49949.3s


In [12]:

scores_df = pd.DataFrame(sentence_scores, columns=['speech_id', 'sentence_position', 'Anti-Elitism', 'People-Centrism', 'Left-Wing Host-Ideology', 'Right-Wing Host-Ideology'])

THRESHOLDS = {
    'Anti-Elitism': 0.415961,
    'People-Centrism': 0.295400,
    'Left-Wing Host-Ideology': 0.429109,
    'Right-Wing Host-Ideology': 0.302714
} # from https://huggingface.co/luerhard/PopBERT

for col, threshold in THRESHOLDS.items():
    scores_df[f'{col}_pred'] = (scores_df[col] >= threshold).astype(int)

all_data_combined = pd.merge(subset, scores_df, on=['speech_id', 'sentence_position'])

In [15]:
all_data_combined.to_csv("all_data_combined_copy.csv", index=False)
# faster_quant = all_data_combined
# slower_quant = all_data_combined

# slower_quant[slower_quant["Anti-Elitism_pred"] == 1]
# faster_quant[faster_quant["Anti-Elitism_pred"] == 1] faster quant performs shittyyyyyy
